# Laboratorio 5. Pipeline funcional integrado

> **Uso en VS Code**: selecciona el kernel del entorno virtual del tema (`.venv`) antes de ejecutar.
>
> Ejecuta las celdas en orden. Puedes modificar valores y volver a ejecutar una celda para observar el resultado.
>
> El código está separado en pasos para reproducir progresivamente el laboratorio del script `.py`.

## 1. Datos de entrada

Se parte de líneas de texto que simulan datos recibidos de un inventario. El formato esperado es `nombre_servicio;puerto;estado`.

In [ ]:
from functools import reduce

lineas = [
    " SSH ; 22 ; activo ",
    " nginx ; 443 ; activo ",
    " api ; 8080 ; detenido ",
    " backup ; 70000 ; activo ",
    " rdp ; 3389 ; activo ",
    " ntp ; 123 ; activo ",
]

print("Datos originales:")
for linea in lineas:
    print(repr(linea))

## 2. Funciones pequeñas para cada paso

Cada función resuelve una tarea concreta del flujo: limpiar, dividir, construir, validar o describir.

In [ ]:
def limpiar_linea(linea):
    """Elimina espacios sobrantes de la línea completa."""
    return linea.strip()


def dividir_linea(linea):
    """
    Divide una línea en campos.

    Returns:
        Tupla con nombre, puerto como texto y estado.
    """
    nombre, puerto, estado = linea.split(";")
    return nombre.strip().lower(), puerto.strip(), estado.strip().lower()


def construir_servicio(campos):
    """
    Convierte una tupla de campos en un diccionario de servicio.

    El puerto se convierte a entero. En este ejemplo los datos son controlados.
    """
    nombre, puerto, estado = campos
    return {
        "nombre": nombre,
        "puerto": int(puerto),
        "activo": estado == "activo",
    }


def puerto_valido(servicio):
    """Devuelve True si el puerto del servicio está en el rango válido."""
    return 1 <= servicio["puerto"] <= 65535


def servicio_activo(servicio):
    """Devuelve True si el servicio está marcado como activo."""
    return servicio["activo"]


def describir_servicio(servicio):
    """Devuelve una cadena legible para el reporte."""
    return f'{servicio["nombre"]}:{servicio["puerto"]}'


print("Funciones auxiliares definidas.")

## 3. Limpiar y convertir los datos

`map()` transforma cada colección en la entrada del siguiente paso.

In [ ]:
limpias = list(map(limpiar_linea, lineas))
campos = list(map(dividir_linea, limpias))
servicios = list(map(construir_servicio, campos))

print("Líneas limpias:", limpias)
print("Campos:", campos)
print("Servicios:", servicios)

## 4. Filtrar servicios activos y puertos válidos

`filter()` permite conservar solo los servicios que cumplen una condición.

In [ ]:
activos = list(filter(servicio_activo, servicios))
activos_validos = list(filter(puerto_valido, activos))

print("Servicios activos:", activos)
print("Servicios activos con puerto válido:", activos_validos)

## 5. Ordenar y construir reporte

`sorted(key=...)` ordena sin modificar la lista original. Después `map()` construye las líneas del reporte.

In [ ]:
ordenados = sorted(activos_validos, key=lambda servicio: servicio["puerto"])
reporte = list(map(describir_servicio, ordenados))

print("Reporte:")
for linea in reporte:
    print("-", linea)

## 6. Validaciones con `any()` y `all()`

`any()` y `all()` permiten validar colecciones sin escribir bucles explícitos.

In [ ]:
hay_puertos_altos = any(servicio["puerto"] > 1024 for servicio in activos_validos)
todos_validos = all(puerto_valido(servicio) for servicio in activos_validos)

print("Hay puertos altos:", hay_puertos_altos)
print("Todos los activos filtrados tienen puertos válidos:", todos_validos)

## 7. Resumen con `reduce()`

Se construye un resumen de estados acumulando resultados en un diccionario.

In [ ]:
def contar_por_estado(conteo, servicio):
    """Acumula cuántos servicios hay por estado lógico."""
    estado = "activo" if servicio["activo"] else "detenido"

    if estado not in conteo:
        conteo[estado] = 0

    conteo[estado] += 1
    return conteo


resumen_estados = reduce(contar_por_estado, servicios, {})

print("Resumen de estados:", resumen_estados)

## 8. Resultado final

El resumen final muestra cuántos elementos entraron, cuántos se interpretaron y qué servicios quedaron listos para reporte.

In [ ]:
print(f"Total de líneas recibidas: {len(lineas)}")
print(f"Total de servicios interpretados: {len(servicios)}")
print(f"Activos con puerto válido: {len(activos_validos)}")
print("Servicios listos para reporte:", reporte)